In [ ]:
#run it once 
%pip install h3

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.1/1.1 MB 1.8 MB/s eta 0:00:00a 0:00:010m

[notice] A new release of pip is available: 23.0.1 -> 26.2.1
[notice] To update, run: pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


In [3]:
import h3
from pyspark.sql import SparkSession
from pyspark.sql.functions import col, monotonically_increasing_id, udf
from pyspark.sql.types import StringType

In [4]:
import os
os.environ["HADOOP_USER_NAME"] = "hdfs"
# 1. Initialize Spark Session
spark = SparkSession.builder \
    .appName("Uber_Medallion_Silver_Layer") \
    .getOrCreate()

Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).


26/09/02 11:04:00 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


In [5]:
# 2. Define H3 Geospatial UDF (Resolution 9)
@udf(returnType=StringType())
def compute_h3(lat, lon, resolution=9):
    if lat is None or lon is None:
        return None
    try:
        # Use h3.latlng_to_cell for h3-py v4+ (fallback to h3.geo_to_h3 if using older versions)
        return h3.latlng_to_cell(lat, lon, resolution)
    except Exception:
        return None

In [ ]:
# 3. Load Bronze Data
df_bronze = spark.read.parquet("hdfs://uber-hadoop-master:9000/data/bronze/rides/*.parquet")

# i have limited the data to 100k , if you crashed limit it to 10k or something
df_limited = df_bronze.limit(100000)



In [ ]:
# 4. Drop Operational Columns & Add Trip ID
# these column are not needed
columns_to_drop = [
    "dispatching_base_num", "originating_base_num", "shared_request_flag", 
    "shared_match_flag", "access_a_ride_flag", "wav_request_flag", "wav_match_flag"
]
df_base = df_limited.drop(*columns_to_drop).withColumn("trip_id", monotonically_increasing_id())

In [8]:
# 5. Define Data Quality Filtration Rules
valid_conditions = (
    col("hvfhs_license_num").isNotNull() &
    col("request_datetime").isNotNull() &
    col("pickup_datetime").isNotNull() &
    col("dropoff_datetime").isNotNull() &
    (col("request_datetime") <= col("pickup_datetime")) &
    (col("pickup_datetime") < col("dropoff_datetime")) &
    col("PULocationID").between(1, 265) &
    col("DOLocationID").between(1, 265) &
    (col("trip_time") > 0) &
    (col("trip_miles") >= 0.0) &
    (col("base_passenger_fare") >= 0.0) &
    (col("tolls") >= 0.0) &
    (col("bcf") >= 0.0) &
    (col("sales_tax") >= 0.0) &
    (col("congestion_surcharge") >= 0.0) &
    (col("airport_fee") >= 0.0) &
    (col("tips") >= 0.0) &
    (col("driver_pay") >= 0.0)
)

In [9]:
# 6. Split Data: Clean vs. Quarantine
df_clean = df_base.filter(valid_conditions)
df_bad_records = df_base.filter(~valid_conditions)

In [10]:
# 7. Write Bad Data to Quarantine (Append mode to keep historical bad records)
df_bad_records.write \
    .mode("append") \
    .parquet("hdfs://uber-hadoop-master:9000/data/quarantine/rides/")

In [11]:
# 8. Load Real Reference Data
df_zones = spark.read.csv("hdfs://uber-hadoop-master:9000/data/reference/zone_centroids.csv", header=True, inferSchema=True)

In [12]:
# 9. Spatial Joins for Coordinates
df_silver = df_clean.join(
    df_zones.withColumnRenamed("LocationID", "PULocationID") \
            .withColumnRenamed("lat", "start_lat") \
            .withColumnRenamed("lon", "start_lon"),
    on="PULocationID",
    how="left"
).join(
    df_zones.withColumnRenamed("LocationID", "DOLocationID") \
            .withColumnRenamed("lat", "end_lat") \
            .withColumnRenamed("lon", "end_lon"),
    on="DOLocationID",
    how="left"
)

In [13]:
# 10. Apply H3 Hashing
df_silver_final = df_silver \
    .withColumn("start_geo_hash", compute_h3(col("start_lat"), col("start_lon"))) \
    .withColumn("end_geo_hash", compute_h3(col("end_lat"), col("end_lon")))

In [14]:
# 11. Write Cleaned, Enriched Data to Silver Layer
df_silver_final.write \
    .mode("overwrite") \
    .parquet("hdfs://uber-hadoop-master:9000/data/silver/staging_rides_geo/")

In [ ]:
# these just to test and see the data in the silver layer
df_silver_check = spark.read.parquet("hdfs://uber-hadoop-master:9000/data/silver/staging_rides_geo/")

df_pandas = df_silver.toPandas()

df_pandas.to_csv("silver_layer_data.csv", index=False)
print("CSV successfully saved to your workspace!")

TypeError: Casting to unit-less dtype 'datetime64' is not supported. Pass e.g. 'datetime64[ns]' instead.